# Task 4: Developing a Flask API for Deep Learning Models

**Objective:** Expose the CNN digit-classification model trained in Task 1 (the from-scratch NumPy CNN, saved as `cnn_digits_model.pkl`) as a REST API using Flask.

**What this notebook does:**
1. Defines the Flask application (`app.py`) with prediction endpoints, JSON request/response handling, and error handling.
2. Starts the API in a background thread (so it can be demonstrated live, inside this notebook).
3. Sends real HTTP requests to every endpoint - including deliberately invalid ones - and shows the actual JSON responses.

The full Flask application is also provided as a standalone file, `app.py`, meant to be run directly with `python app.py` and called from any HTTP client (curl, Postman, a web or mobile front-end, etc.) - running it inside this notebook is done purely for a self-contained, runnable demonstration.

## 1. The Flask Application (`app.py`)

In [1]:
"""
Task 4: Developing a Flask API for Deep Learning Models
==========================================================
Exposes the CNN digit-classification model trained in Task 1 (a from-scratch
NumPy CNN, saved as cnn_digits_model.pkl) as a REST API using Flask.

Endpoints:
  GET  /              - API information
  GET  /health        - health check
  POST /predict       - predict the digit for a single 8x8 grayscale image
  POST /predict/batch - predict digits for multiple images in one request

Run with:  python app.py
Then send requests to http://127.0.0.1:5000/
"""

import os
import pickle
import traceback
import numpy as np
from flask import Flask, request, jsonify

try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # __file__ is not defined when this code runs inside a notebook cell
    # (Jupyter/Colab) rather than as a standalone script - fall back to the
    # current working directory in that case.
    BASE_DIR = os.getcwd()
MODEL_PATH = os.path.join(BASE_DIR, "cnn_digits_model.pkl")

app = Flask(__name__)

# ----------------------------------------------------------------------
# Model definition (must match the architecture used to train and save
# cnn_digits_model.pkl in Task 1) and inference-only forward pass.
# ----------------------------------------------------------------------


def im2col(x, kh, kw, stride=1, pad=0):
    N, H, W, C = x.shape
    if pad > 0:
        x = np.pad(x, ((0, 0), (pad, pad), (pad, pad), (0, 0)))
    out_h = (H + 2 * pad - kh) // stride + 1
    out_w = (W + 2 * pad - kw) // stride + 1
    cols = np.zeros((N, out_h, out_w, kh, kw, C), dtype=x.dtype)
    for i in range(kh):
        i_max = i + stride * out_h
        for j in range(kw):
            j_max = j + stride * out_w
            cols[:, :, :, i, j, :] = x[:, i:i_max:stride, j:j_max:stride, :]
    return cols.reshape(N, out_h, out_w, kh * kw * C), out_h, out_w


def conv_forward(x, W, b, stride=1, pad=1):
    N, H, Wd, C = x.shape
    k = W.shape[0]
    out_ch = W.shape[-1]
    cols, out_h, out_w = im2col(x, k, k, stride, pad)
    W_col = W.reshape(-1, out_ch)
    out = cols.reshape(N * out_h * out_w, -1) @ W_col + b
    return out.reshape(N, out_h, out_w, out_ch)


def relu(x):
    return np.maximum(0, x)


def maxpool_forward(x, size=2, stride=2):
    N, H, W, C = x.shape
    out_h, out_w = H // stride, W // stride
    x = x[:, :out_h * stride, :out_w * stride, :]
    out = np.zeros((N, out_h, out_w, C), dtype=x.dtype)
    for i in range(out_h):
        for j in range(out_w):
            window = x[:, i*stride:i*stride+size, j*stride:j*stride+size, :]
            out[:, i, j, :] = window.max(axis=(1, 2))
    return out


def softmax(logits):
    e = np.exp(logits - logits.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


class DigitCNN:
    """Loads the Task 1 CNN weights and runs inference only (no training)."""

    def __init__(self, weights_path):
        with open(weights_path, "rb") as f:
            self.w = pickle.load(f)

    def predict_proba(self, x):
        """x: numpy array of shape (N, 8, 8, 1), pixel values already in [0, 1]."""
        w = self.w
        x = conv_forward(x, w["conv1_W"], w["conv1_b"], stride=1, pad=1)
        x = relu(x)
        x = maxpool_forward(x, 2, 2)
        x = conv_forward(x, w["conv2_W"], w["conv2_b"], stride=1, pad=1)
        x = relu(x)
        x = maxpool_forward(x, 2, 2)
        x = x.reshape(x.shape[0], -1)
        x = x @ w["fc1_W"] + w["fc1_b"]
        x = relu(x)
        logits = x @ w["fc2_W"] + w["fc2_b"]
        return softmax(logits)


# Load the model once at startup, not on every request.
model = DigitCNN(MODEL_PATH)


# ----------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------

def parse_image(payload):
    """
    Validates and converts a single image payload into a (1, 8, 8, 1)
    float32 array normalized to [0, 1]. Raises ValueError with a clear
    message on any problem, which the route handlers turn into a 400.
    """
    if payload is None:
        raise ValueError("Missing 'image' field in request body.")

    arr = np.array(payload, dtype=np.float64)

    if arr.size != 64:
        raise ValueError(
            f"Expected an 8x8 (64-value) grayscale image, got {arr.size} values."
        )

    arr = arr.reshape(8, 8)

    if np.isnan(arr).any():
        raise ValueError("Image contains non-numeric or missing values.")

    # Accept either raw 0-16 pixel scale (like the original digits dataset)
    # or already-normalized 0-1 scale, and normalize consistently to [0, 1].
    if arr.max() > 1.0:
        if arr.max() > 16.0 or arr.min() < 0.0:
            raise ValueError("Pixel values must be within [0, 16] (or already normalized to [0, 1]).")
        arr = arr / 16.0
    elif arr.min() < 0.0:
        raise ValueError("Pixel values must not be negative.")

    return arr.reshape(1, 8, 8, 1).astype(np.float32)


def format_prediction(probs_row):
    probs_row = probs_row.tolist()
    pred_digit = int(np.argmax(probs_row))
    return {
        "predicted_digit": pred_digit,
        "confidence": round(probs_row[pred_digit], 4),
        "probabilities": {str(i): round(p, 4) for i, p in enumerate(probs_row)},
    }


# ----------------------------------------------------------------------
# Routes
# ----------------------------------------------------------------------

@app.route("/", methods=["GET"])
def index():
    return jsonify({
        "name": "Digit Classification API",
        "description": "REST API serving the Task 1 from-scratch CNN digit classifier.",
        "endpoints": {
            "GET /health": "Health check.",
            "POST /predict": "Predict the digit for one 8x8 grayscale image. "
                              "Body: {\"image\": [64 numbers, 0-16 or 0-1, row-major 8x8]}",
            "POST /predict/batch": "Predict digits for multiple images. "
                                    "Body: {\"images\": [[64 numbers], [64 numbers], ...]}",
        },
    })


@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "model_loaded": model is not None})


@app.route("/predict", methods=["POST"])
def predict():
    try:
        data = request.get_json(silent=True)
        if data is None:
            return jsonify({"error": "Request body must be valid JSON."}), 400

        image = parse_image(data.get("image"))
        probs = model.predict_proba(image)[0]
        return jsonify(format_prediction(probs)), 200

    except ValueError as e:
        return jsonify({"error": str(e)}), 400
    except Exception as e:
        app.logger.error("Unexpected error in /predict: %s\n%s", e, traceback.format_exc())
        return jsonify({"error": "Internal server error while generating prediction."}), 500


@app.route("/predict/batch", methods=["POST"])
def predict_batch():
    try:
        data = request.get_json(silent=True)
        if data is None:
            return jsonify({"error": "Request body must be valid JSON."}), 400

        images_payload = data.get("images")
        if not isinstance(images_payload, list) or len(images_payload) == 0:
            return jsonify({"error": "'images' must be a non-empty list of 64-value images."}), 400
        if len(images_payload) > 100:
            return jsonify({"error": "Batch size limited to 100 images per request."}), 400

        results = []
        for idx, img_payload in enumerate(images_payload):
            try:
                image = parse_image(img_payload)
                probs = model.predict_proba(image)[0]
                results.append(format_prediction(probs))
            except ValueError as e:
                results.append({"error": f"image[{idx}]: {e}"})

        return jsonify({"count": len(results), "results": results}), 200

    except Exception as e:
        app.logger.error("Unexpected error in /predict/batch: %s\n%s", e, traceback.format_exc())
        return jsonify({"error": "Internal server error while generating predictions."}), 500


# ----------------------------------------------------------------------
# Error handlers for common HTTP errors
# ----------------------------------------------------------------------

@app.errorhandler(404)
def not_found(e):
    return jsonify({"error": "Endpoint not found. See GET / for a list of available endpoints."}), 404


@app.errorhandler(405)
def method_not_allowed(e):
    return jsonify({"error": "Method not allowed for this endpoint."}), 405


@app.errorhandler(500)
def server_error(e):
    return jsonify({"error": "Internal server error."}), 500

# Note: the standalone app.py file ends with:
#     if __name__ == "__main__":
#         app.run(host="0.0.0.0", port=5000, debug=False)
# That line is intentionally left out here: __name__ is also "__main__"
# inside a notebook cell, so including it would call the BLOCKING Flask
# dev server immediately and freeze this cell forever. The next cell
# starts the server properly, in a background thread, instead.


## 2. Start the API (in a background thread, for this notebook demo)

**Important - about opening the API in a browser:** if this notebook is running on a remote machine (e.g. Google Colab), `http://127.0.0.1:5050` refers to *that remote machine*, not your own laptop - so pasting that URL into your own browser's address bar will not connect to anything. This is expected, not an error. All the cells below call the API correctly, from Python, using the `requests` library, from inside the same runtime the server is running on - that always works, on Colab or anywhere else.

If you specifically want a real, public URL you can open in your own browser, see the optional `pyngrok` cell right after this one.

In [1]:
import threading, time

def run_server():
    app.run(host="127.0.0.1", port=5050, debug=False, use_reloader=False)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(1.5)
print("API is running on http://127.0.0.1:5050")

API is running on http://127.0.0.1:5050


In [1]:
# OPTIONAL - only needed if you want a real, public URL you can open in
# your own browser (not required to run the rest of this notebook).
#
# !pip install pyngrok --quiet
# from pyngrok import ngrok
# public_url = ngrok.connect(5050)
# print("Public URL:", public_url)

### (Optional) Expose the API with a public URL

The cell above is commented out by default since it needs a free ngrok account/token and isn't required for anything else in this notebook. Uncomment it, install `pyngrok`, and run it if you want a URL you can actually paste into your own browser or share with someone else.

## 3. API Information and Health Check

In [1]:
import requests, json

resp = requests.get("http://127.0.0.1:5050/")
print(json.dumps(resp.json(), indent=2))

{
  "description": "REST API serving the Task 1 from-scratch CNN digit classifier.",
  "endpoints": {
    "GET /health": "Health check.",
    "POST /predict": "Predict the digit for one 8x8 grayscale image. Body: {\"image\": [64 numbers, 0-16 or 0-1, row-major 8x8]}",
    "POST /predict/batch": "Predict digits for multiple images. Body: {\"images\": [[64 numbers], [64 numbers], ...]}"
  },
  "name": "Digit Classification API"
}


In [1]:
resp = requests.get("http://127.0.0.1:5050/health")
print(json.dumps(resp.json(), indent=2))

{
  "model_loaded": true,
  "status": "ok"
}


## 4. Single Prediction

Sending a hand-written-style '0' pattern (8x8 grayscale, 0-16 pixel scale, the same format as the raw scikit-learn Digits dataset used in Task 1).

In [1]:
import matplotlib.pyplot as plt
import numpy as np

sample_zero = [
    0, 0, 5, 13, 9, 1, 0, 0,
    0, 0, 13, 15, 10, 15, 5, 0,
    0, 3, 15, 2, 0, 11, 8, 0,
    0, 4, 12, 0, 0, 8, 8, 0,
    0, 5, 8, 0, 0, 9, 8, 0,
    0, 4, 11, 0, 1, 12, 7, 0,
    0, 2, 14, 5, 10, 12, 0, 0,
    0, 0, 6, 13, 10, 0, 0, 0,
]

# Show the actual image being sent to the API, so the prediction below
# can be checked against it visually.
plt.figure(figsize=(2.5, 2.5))
plt.imshow(np.array(sample_zero).reshape(8, 8), cmap="gray")
plt.title("Image sent to /predict")
plt.axis("off")
plt.show()

resp = requests.post("http://127.0.0.1:5050/predict", json={"image": sample_zero})
print(json.dumps(resp.json(), indent=2))

## 5. Batch Prediction on Real Digit Images

Three genuine images from the scikit-learn Digits dataset (true labels 3, 7, and 8) sent together in a single batch request.

In [1]:
from sklearn.datasets import load_digits

d = load_digits()

def get_sample(digit):
    idx = list(d.target).index(digit)
    return d.images[idx]

true_labels = [3, 7, 8]
sample_images = [get_sample(t) for t in true_labels]

fig, axes = plt.subplots(1, 3, figsize=(7, 2.6))
for ax, img, label in zip(axes, sample_images, true_labels):
    ax.imshow(img, cmap="gray")
    ax.set_title(f"True label: {label}", fontsize=11)
    ax.axis("off")
plt.suptitle("Images Sent to /predict/batch", fontsize=12)
plt.tight_layout()
plt.show()

In [1]:
from sklearn.datasets import load_digits
d = load_digits()

def get_sample(digit):
    idx = list(d.target).index(digit)
    return d.images[idx].flatten().tolist()

images = [get_sample(3), get_sample(7), get_sample(8)]
resp = requests.post("http://127.0.0.1:5050/predict/batch", json={"images": images})
print(json.dumps(resp.json(), indent=2))

{
  "count": 3,
  "results": [
    {
      "confidence": 0.9998,
      "predicted_digit": 3,
      "probabilities": {
        "0": 0.0,
        "1": 0.0,
        "2": 0.0,
        "3": 0.9998,
        "4": 0.0,
        "5": 0.0,
        "6": 0.0,
        "7": 0.0,
        "8": 0.0,
        "9": 0.0001
      }
    },
    {
      "confidence": 0.9827,
      "predicted_digit": 7,
      "probabilities": {
        "0": 0.0,
        "1": 0.0007,
        "2": 0.0009,
        "3": 0.0024,
        "4": 0.0022,
        "5": 0.0009,
        "6": 0.0,
        "7": 0.9827,
        "8": 0.0012,
        "9": 0.009
      }
    },
    {
      "confidence": 0.9992,
      "predicted_digit": 8,
      "probabilities": {
        "0": 0.0,
        "1": 0.0,
        "2": 0.0,
        "3": 0.0006,
        "4": 0.0,
        "5": 0.0,
        "6": 0.0001,
        "7": 0.0,
        "8": 0.9992,
        "9": 0.0001
      }
    }
  ]
}


## 6. Error Handling - Deliberately Invalid Requests

In [1]:
# Missing 'image' field
resp = requests.post("http://127.0.0.1:5050/predict", json={})
print("Status code:", resp.status_code)
print(json.dumps(resp.json(), indent=2))

Status code: 400
{
  "error": "Missing 'image' field in request body."
}


In [1]:
# Wrong-size image array
resp = requests.post("http://127.0.0.1:5050/predict", json={"image": [1, 2, 3]})
print("Status code:", resp.status_code)
print(json.dumps(resp.json(), indent=2))

Status code: 400
{
  "error": "Expected an 8x8 (64-value) grayscale image, got 3 values."
}


In [1]:
# Unknown endpoint -> 404
resp = requests.get("http://127.0.0.1:5050/nonexistent")
print("Status code:", resp.status_code)
print(json.dumps(resp.json(), indent=2))

Status code: 404
{
  "error": "Endpoint not found. See GET / for a list of available endpoints."
}


In [1]:
# Wrong HTTP method -> 405
resp = requests.get("http://127.0.0.1:5050/predict")
print("Status code:", resp.status_code)
print(json.dumps(resp.json(), indent=2))

Status code: 405
{
  "error": "Method not allowed for this endpoint."
}


## 7. Observations

- All three real test digits (true labels 3, 7, 8) were classified correctly by the API, with confidence between 98.3% and 99.9% - consistent with the ~97% test accuracy measured for this model in Task 1.
- Every error case (missing field, wrong-size array, unknown endpoint, wrong HTTP method) returns a clean, structured JSON error message with an appropriate HTTP status code (400, 404, or 405) instead of a raw server stack trace, which is what makes the API safe to call from an untrusted client.
- The batch endpoint processes multiple images in a single HTTP round-trip and isolates failures per-image, so one malformed image in a batch does not fail the other, valid images in the same request.
- The model is loaded once when the Flask app starts, not on every request, so predictions are returned quickly (well under the time it took to train the model itself).

## 8. Files Produced
- `app.py` - the Flask application (runnable standalone with `python app.py`)
- `README.md` - full API documentation (endpoints, request/response formats, error codes, usage examples)
- `sample_client.py` - a standalone script demonstrating calls to every endpoint with the `requests` library
- `requirements.txt` - dependencies needed to run the API